# VirtualiZarr frictions on large single-cell HDF5 (`.h5ad`)

VirtualiZarr grew up on **many small-to-medium NetCDF/HDF5 files** on a **regular chunk grid**, stacked along a new (time) axis. Single-cell genomics (`.h5ad`, an HDF5 file) is the opposite regime:

- **one large, singular file** (10–40+ GB), not a stack of small ones;
- the main array is a **sparse CSR matrix** stored as three independent 1-D datasets (`X/data`, `X/indices`, `X/indptr`), not a rectangular grid;
- it is **chunk-dense**: `X/data` for an 815k-cell atlas has ~107k chunks, `X/indices` ~215k; a 3.76M-cell atlas has millions.

This notebook reproduces two frictions on a **synthetic 8 MB file** (so it runs anywhere). Every table below is generated by the cell above it.

In [1]:
import functools, io, tempfile, warnings
from pathlib import Path
import numpy as np, pandas as pd, h5py
warnings.filterwarnings("ignore")

from obspec_utils.registry import ObjectStoreRegistry
from obstore.store import LocalStore
from obspec_utils.readers import BlockStoreReader
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import HDFParser

tmp = tempfile.mkdtemp()
PATH = str(Path(tmp) / "dense.h5")

def make_file(path, n=4_000_000, chunk=1_000):
    """One 1-D gzip dataset with many chunks -> a large, scattered chunk B-tree."""
    rng = np.random.default_rng(0)
    data = (rng.standard_normal(n) * 100).astype("int64")
    with h5py.File(path, "w") as f:
        f.create_dataset("x", data=data, chunks=(chunk,), compression="gzip")

make_file(PATH)
size = Path(PATH).stat().st_size
with h5py.File(PATH) as f:
    nchunks = f["x"].id.get_num_chunks()
print(f"file: {size/1e6:.1f} MB   dataset x: {nchunks} chunks")
print(f"a manifest needs only offset+length per chunk = {nchunks*16/1e3:.0f} KB of metadata")

file: 8.1 MB   dataset x: 4000 chunks
a manifest needs only offset+length per chunk = 64 KB of metadata


## Friction 1 — building a *virtual* manifest reads ~the whole file

Building a virtual reference should touch only metadata: per chunk, its byte offset + length (HDF5 keeps these in the dataset's B-tree). It should **not** read chunk data. First, what does a native h5py B-tree walk actually read? We route h5py through a logging file object and record every `(offset, nbytes)`.

In [2]:
class LoggingFile(io.RawIOBase):
    """h5py reads through this; we record every read's size."""
    def __init__(self, path):
        self._f = open(path, "rb"); self.log = []
    def readable(self): return True
    def seekable(self): return True
    def seek(self, o, w=0): return self._f.seek(o, w)
    def tell(self): return self._f.tell()
    def read(self, n=-1):
        off = self._f.tell(); b = self._f.read(n); self.log.append((off, len(b))); return b
    def readinto(self, buf):
        off = self._f.tell(); d = self._f.read(len(buf)); buf[:len(d)] = d
        self.log.append((off, len(d))); return len(d)

lf = LoggingFile(PATH)
with h5py.File(lf) as f:
    dsid = f["x"].id
    n = dsid.get_num_chunks()
    dsid.chunk_iter(lambda blob: None)  # walk the whole B-tree

sizes = np.array([s for _, s in lf.log])
print(f"native h5py walk of {n} chunks:")
print(f"  reads (requests) : {len(lf.log)}")
print(f"  bytes read       : {sizes.sum()/1e6:.2f} MB")
print(f"  typical read size: {np.median(sizes):.0f} bytes  (one B-tree node)")
print(f"  B-tree entries per read: {n/len(lf.log):.1f}  <- one small read yields many chunks")

native h5py walk of 4000 chunks:
  reads (requests) : 82
  bytes read       : 0.16 MB
  typical read size: 2096 bytes  (one B-tree node)
  B-tree entries per read: 48.8  <- one small read yields many chunks


Each read is a single small B-tree node packing many chunk records — so the walk needs far fewer reads than there are chunks. Now the same manifest built by VirtualiZarr's `HDFParser`, which reads the file through obspec's `BlockStoreReader` (default **1 MiB blocks, 64 MiB LRU**). We count bytes fetched *at the store* (cache-independent) with a tallying proxy, sweeping the block size.

In [3]:
class TallyStore:
    """Delegates to a real LocalStore, tallying bytes returned by range reads."""
    def __init__(self, inner): self._inner = inner; self.bytes = 0; self.reqs = 0
    def get_ranges(self, path, starts, lengths):
        r = self._inner.get_ranges(path, starts=starts, lengths=lengths)
        self.bytes += sum(int(x) for x in lengths); self.reqs += len(list(lengths)); return r
    def get_range(self, path, *, start, length):
        r = self._inner.get_range(path, start=start, length=length)
        self.bytes += int(length); self.reqs += 1; return r
    def __getattr__(self, k): return getattr(self._inner, k)

def parser_read(factory):
    t = TallyStore(LocalStore(prefix=tmp))
    reg = ObjectStoreRegistry({f"file://{tmp}": t})
    vds = open_virtual_dataset(url=f"file://{PATH}", registry=reg,
                               parser=HDFParser(reader_factory=factory))
    _ = vds["x"].data.manifest
    return t.reqs, t.bytes

rows = [("native h5py chunk_iter", len(lf.log), sizes.sum())]
for kb in (8, 64, 1024):
    reqs, b = parser_read(functools.partial(BlockStoreReader, block_size=kb * 1024))
    label = f"HDFParser block={kb} KiB" + (" (default)" if kb == 1024 else "")
    rows.append((label, reqs, b))

df = pd.DataFrame(rows, columns=["approach", "requests", "bytes"])
df["MB fetched"] = (df["bytes"] / 1e6).round(2)
df["x over metadata"] = (df["bytes"] / (nchunks * 16)).round(0).astype(int)
df["% of file"] = (100 * df["bytes"] / size).round(0).astype(int)
df[["approach", "requests", "MB fetched", "x over metadata", "% of file"]]

,approach,requests,MB fetched,x over metadata,% of file
0,native h5py chunk_iter,82,0.16,2,2
1,HDFParser block=8 KiB,91,0.75,12,9
2,HDFParser block=64 KiB,73,4.73,74,59
3,HDFParser block=1024 KiB (default),8,8.07,126,100


The default (1 MiB) reads ~**100% of the file** to recover a few KB of offsets — here in a handful of large requests; shrinking the block cuts bytes but multiplies requests. On this small file the whole B-tree fits in a few blocks, so the cost is almost entirely **wasted bytes**.

At real scale the picture is worse on *both* axes. On an 815k-cell atlas (`X/data`, 107,508 chunks, 10.3 GB file) the block reader's 64 MiB LRU thrashes over the large span and re-fetches revisited internal B-tree nodes, so it loses on request count too:

| approach | requests | bytes fetched |
|---|---:|---:|
| native h5py `chunk_iter` | **3,855** | **8.1 MB** |
| HDFParser `block=8 KiB` | 11,999 | 98 MB |
| HDFParser `block=64 KiB` | 9,750 | 639 MB |
| HDFParser `block=1 MiB` (default) | 7,793 | **8.17 GB** |

Native is **below the trade-off curve on both axes** — it doesn't re-fetch revisited nodes and doesn't quantize 2 KiB reads up to blocks. Tuning `block_size` only trades bytes for more requests; it never reaches native.

**This is largely a default-configuration pitfall** (1 MiB is tuned for cloud object stores, the wrong regime for a scattered metadata walk) — but there is no single good block size, and the walk is *serial and dependent* (node N+1's location is only known after reading node N), so no generic reader parallelizes it.

### The fix: build the manifest from a natively-opened dataset

VirtualiZarr's own `_construct_manifest_array`, handed a dataset from a native `h5py.File`, produces a **byte-identical** `ManifestArray` without routing metadata reads through the block reader.

In [4]:
from virtualizarr.parsers.hdf.hdf import _construct_manifest_array

t = TallyStore(LocalStore(prefix=tmp))
reg = ObjectStoreRegistry({f"file://{tmp}": t})
parser_ma = open_virtual_dataset(url=f"file://{PATH}", registry=reg, parser=HDFParser())["x"].data

with h5py.File(PATH) as f:
    native_ma = _construct_manifest_array(filepath=f"file://{PATH}", dataset=f["x"], group="/")

same = (
    parser_ma.shape == native_ma.shape
    and parser_ma.chunks == native_ma.chunks
    and parser_ma.dtype == native_ma.dtype
    and int(parser_ma.manifest._offsets[(0,)]) == int(native_ma.manifest._offsets[(0,)])
    and int(parser_ma.manifest._lengths[(0,)]) == int(native_ma.manifest._lengths[(0,)])
)
print(f"native ManifestArray: shape={native_ma.shape} chunks={native_ma.chunks} dtype={native_ma.dtype}")
print(f"codecs={[type(c).__name__ for c in native_ma.metadata.codecs]}  fill={native_ma.metadata.fill_value}")
print(f"identical to HDFParser's manifest (shape/chunks/dtype/offset/length): {same}")

native ManifestArray: shape=(4000000,) chunks=(1000,) dtype=int64
codecs=['BytesCodec', 'Zlib']  fill=0
identical to HDFParser's manifest (shape/chunks/dtype/offset/length): True


For **remote** sources the walk is still ~thousands of serial 2 KiB reads per file; over HTTP that is latency-bound whatever the reader (fsspec's default 5 MiB read-ahead just re-introduces the amplification). So the robust pattern is **download-to-local once** (bandwidth-bound, parallel), then walk natively.

**Upstream directions:** a smaller / HDF-aware default `block_size`; an HDF-metadata-aware reader (or a fast path that builds the `ChunkManifest` via `chunk_iter` on a local file); optionally a two-phase B-tree fetch (internal nodes, then batch leaves concurrently) for the remote case.

## Friction 2 — ragged concat isn't expressible; independent 1-D arrays fight the dimension model

**2a. Concat requires a uniform chunk grid.** Each atlas chunks `X/data` differently (HDF5 sizes chunks per file). VirtualiZarr's concat refuses:

```
ValueError: Cannot concatenate arrays with inconsistent chunk shapes:
(5973,) vs (5939,). Requires ZEP003 (Variable-length Chunks).
```

This is a regular-grid assumption, not a real wall: zarr-python ships rectilinear (variable-length) chunk grids and Icechunk stores refs by chunk coordinate regardless of grid. We bypass the concat and assemble the combined array by hand (merged grid = every file's HDF5 edges end-to-end; each full chunk → a virtual ref; each file's HDF5-padded tail chunk → one native chunk of its valid elements). It works but is ~200 lines of manifest surgery that arguably belongs upstream as first-class **ragged concat**.

**2b. Independent 1-D arrays share a phony dimension.** The three CSR arrays are independent 1-D buffers of different lengths, but `open_virtual_dataset` funnels through an xarray `Dataset` that auto-names each first axis `phony_dim_0` and rejects the conflicting lengths — the cell below reproduces it.

In [5]:
# A CSR-like trio of independent 1-D datasets of different lengths in one group.
csr_path = str(Path(tmp) / "csr.h5")
with h5py.File(csr_path, "w") as f:
    g = f.create_group("X")
    g.create_dataset("data", data=np.ones(2_286_884, "float32"), chunks=(8934,))
    g.create_dataset("indices", data=np.ones(2_286_884, "int32"), chunks=(8934,))
    g.create_dataset("indptr", data=np.arange(2701, dtype="int32"), chunks=(2701,))

reg = ObjectStoreRegistry({f"file://{tmp}": LocalStore(prefix=tmp)})
try:
    open_virtual_dataset(url=f"file://{csr_path}", registry=reg, parser=HDFParser(group="X"))
    print("opened (unexpected)")
except Exception as e:
    print(f"{type(e).__name__}: {e}")
    print("\n-> data/indices (len 2286884) and indptr (len 2701) both get 'phony_dim_0';")
    print("   they are independent buffers with no shared dimension. We work around it by")
    print("   opening one array at a time with siblings dropped, then assembling by hand.")

ValueError: conflicting sizes for dimension 'phony_dim_0': length 2701 on 'indptr' and length 2286884 on {'phony_dim_0': 'data'}

-> data/indices (len 2286884) and indptr (len 2701) both get 'phony_dim_0';
   they are independent buffers with no shared dimension. We work around it by
   opening one array at a time with siblings dropped, then assembling by hand.


## Takeaway

Both frictions are the same story: assumptions that are free for many-small-regular-files (route all reads through one block reader; every array shares a grid and a dimension) become expensive or blocking for a single large, chunk-dense, sparse-encoded HDF5. Single-cell atlases (10⁵–10⁷ cells) are a large, growing corpus that would virtualize cleanly if these were addressed; today it takes a hand-written manifest layer to get there.

Companion prose (for an issue): [`virtualizarr-single-cell-frictions.md`](./virtualizarr-single-cell-frictions.md). Standalone repro: [`vz_hdf_manifest_read_amplification.py`](./vz_hdf_manifest_read_amplification.py).